# Untuned YOLOv8m Single Year Baseline

This notebook provides the final untuned YOLOv8m baseline for the single year weed detection experiments.

The protocol is intentionally fixed. It uses the same YOLOv8m model scale, image size, batch size, worker count, cache mode, full fidelity epoch count, training seeds, and non tuned training settings used by the newer optimization notebooks. The five reference hyperparameters remain at their untuned values.

The test split is never used for model selection. Each run trains on the training split, uses the validation split during training, freezes `best.pt`, and evaluates that checkpoint once on the test split.


## Module 1: Imports, experiment protocol, paths, and fixed settings

This module defines every shared variable used later in the notebook. Change `MAIN_DIRECTORY` and `DATASET_YEAR` only when moving the experiment or switching between Year2021 and Year2022.


In [ ]:
from pathlib import Path
from datetime import datetime
from statistics import mean, stdev
from typing import Dict, Any, List, Optional, Tuple
from collections import Counter

import csv
import json
import math
import os
import random
import shutil
import time
import uuid
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image, ImageDraw, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO


# Experiment location
MAIN_DIRECTORY = Path(r"C:\Environments\YoloWeeds")
DATASET_YEAR = "2022"
DATASET_TAG = f"Year{DATASET_YEAR}"
SOURCE_DATASET_ROOT = MAIN_DIRECTORY / "Datasets" / DATASET_TAG

# Persistent experiment outputs
PROJECT_ROOT = MAIN_DIRECTORY / f"Weed_{DATASET_TAG}_Untuned_YOLOv8m"
RUNS_ROOT = PROJECT_ROOT / "runs"
SPLITS_ROOT = PROJECT_ROOT / "splits"
STATS_ROOT = PROJECT_ROOT / "stats"
YOLO_DATA_ROOT = PROJECT_ROOT / "yolo_data"

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUNS_ROOT / "untuned_yolov8m" / RUN_TIMESTAMP
TRAINING_ROOT = RUN_ROOT / "training"
TEST_EVAL_ROOT = RUN_ROOT / "test_eval"
RUN_STATS_ROOT = RUN_ROOT / "stats"
RUN_PLOTS_ROOT = RUN_ROOT / "plots"

# Defined for cleanup compatibility. This baseline does not perform HPO.
HPO_RUNS_ROOT = RUN_ROOT / "hpo"

for folder in [
    PROJECT_ROOT,
    RUNS_ROOT,
    SPLITS_ROOT,
    STATS_ROOT,
    RUN_ROOT,
    TRAINING_ROOT,
    TEST_EVAL_ROOT,
    RUN_STATS_ROOT,
    RUN_PLOTS_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)


# Target classes
CLASS_NAMES = [
    "Waterhemp",
    "Carpetweed",
    "Morningglory",
    "Goosegrass",
    "SpottedSpurge",
    "PalmerAmaranth",
    "Purslane",
    "Ragweed",
]
TARGET_CLASS_SET = set(CLASS_NAMES)

LABEL_ALIASES = {
    "morninglory": "Morningglory",
    "morning glory": "Morningglory",
    "morningglory": "Morningglory",
    "spotted spurge": "SpottedSpurge",
    "spottedspurge": "SpottedSpurge",
    "palmer amaranth": "PalmerAmaranth",
    "palmeramaranth": "PalmerAmaranth",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


# Split protocol
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15
FINAL_EVALUATION_SEEDS = [42, 123, 999]
REQUIRE_ALL_CLASSES_PER_SPLIT = True
REQUIRE_ALL_FINAL_RUNS = True


# Shared YOLO runtime protocol copied from the newer experiment
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 8
PATIENCE = 0
IMAGE_PLACEMENT_MODE = "symlink"
CACHE_MODE = "disk"

FULL_FIDELITY_EPOCHS = 40
FULL_FIDELITY_WARMUP_EPOCHS = 3.0
FULL_FIDELITY_CLOSE_MOSAIC = 10

# These are reference untuned values, not values selected by HPO.
UNTUNED_HYPERPARAMETERS = {
    "lr0": 0.010,
    "lrf": 0.010,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "scale": 0.50,
}

FIXED_TRAINING_ARGS = {
    "optimizer": "SGD",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "nbs": 64,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.70,
    "hsv_v": 0.40,
    "degrees": 0.0,
    "translate": 0.10,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mixup": 0.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cache": CACHE_MODE,
    "cos_lr": False,
    "rect": False,
}

STRICT_FITNESS_ALIGNMENT_CHECK = True
FITNESS_ALIGNMENT_TOLERANCE = 0.002

BASELINE_EPOCH_UNITS = len(FINAL_EVALUATION_SEEDS) * FULL_FIDELITY_EPOCHS
assert BASELINE_EPOCH_UNITS == 120


print("=" * 92)
print("UNTUNED YOLOv8m SINGLE YEAR BASELINE")
print("=" * 92)
print("Dataset:", DATASET_TAG)
print("Dataset root:", SOURCE_DATASET_ROOT)
print("Model:", MODEL_NAME)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Workers:", WORKERS)
print("Cache:", CACHE_MODE)
print("Patience:", PATIENCE)
print("Epochs per run:", FULL_FIDELITY_EPOCHS)
print("Seeds:", FINAL_EVALUATION_SEEDS)
print("Total baseline epoch units:", BASELINE_EPOCH_UNITS)
print("Untuned hyperparameters:", json.dumps(UNTUNED_HYPERPARAMETERS, indent=2))
print("Python:", os.sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Run root:", RUN_ROOT)
print("=" * 92)


## Module 2: Annotation parsing helpers

This module reads Pascal VOC XML and common JSON annotations, normalizes known label aliases, checks image dimensions when needed, and clips bounding boxes to valid image boundaries.


In [ ]:
def normalize_label(raw_label: str, label_aliases: Dict[str, str]) -> str:
    """Normalize one class label using the supplied alias table."""
    cleaned = str(raw_label).strip()
    return label_aliases.get(cleaned.lower(), cleaned)


def read_image_size(image_path: Path) -> Tuple[int, int]:
    """Return image width and height."""
    try:
        with Image.open(image_path) as image:
            return image.size
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError(f"Unreadable image: {image_path}\n{error}") from error


def parse_pascal_voc_xml(
    xml_path: Path,
    label_aliases: Dict[str, str],
) -> Tuple[Optional[Tuple[int, int]], List[Dict[str, Any]]]:
    """Parse one Pascal VOC XML annotation."""
    root = ET.parse(xml_path).getroot()
    annotation_size = None

    size_node = root.find("size")
    if size_node is not None:
        width_text = size_node.findtext("width")
        height_text = size_node.findtext("height")
        if width_text and height_text:
            annotation_size = (int(float(width_text)), int(float(height_text)))

    objects: List[Dict[str, Any]] = []

    for object_node in root.findall("object"):
        raw_name = object_node.findtext("name")
        box_node = object_node.find("bndbox")

        if not raw_name or box_node is None:
            continue

        coordinates = [
            box_node.findtext("xmin"),
            box_node.findtext("ymin"),
            box_node.findtext("xmax"),
            box_node.findtext("ymax"),
        ]

        if any(value is None for value in coordinates):
            continue

        try:
            x1, y1, x2, y2 = [float(value) for value in coordinates]
        except (TypeError, ValueError):
            continue

        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue

        objects.append({
            "label": normalize_label(raw_name, label_aliases),
            "bbox_xyxy": [x1, y1, x2, y2],
        })

    return annotation_size, objects


def extract_via_label(
    attributes: Dict[str, Any],
    label_aliases: Dict[str, str],
) -> Optional[str]:
    """Extract a label from common VIA region attribute layouts."""
    preferred_keys = [
        "label",
        "class",
        "class_name",
        "name",
        "species",
        "type",
        "weed",
        "category",
    ]

    for key in preferred_keys:
        value = attributes.get(key)

        if isinstance(value, str) and value.strip():
            return normalize_label(value, label_aliases)

        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key, label_aliases)

            if len(value) == 1:
                return normalize_label(next(iter(value.keys())), label_aliases)

    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalize_label(value, label_aliases)

    return None


def parse_json_annotation(
    json_path: Path,
    label_aliases: Dict[str, str],
) -> List[Dict[str, Any]]:
    """Parse LabelMe style or VIA style JSON annotations."""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    objects: List[Dict[str, Any]] = []

    if isinstance(data, dict) and isinstance(data.get("shapes"), list):
        for shape in data["shapes"]:
            label = normalize_label(shape.get("label", ""), label_aliases)
            points = shape.get("points", [])

            if not label or len(points) < 2:
                continue

            xs = [float(point[0]) for point in points]
            ys = [float(point[1]) for point in points]
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            if x2 > x1 + 1 and y2 > y1 + 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x1, y1, x2, y2],
                })

        return objects

    metadata = data.get("_via_img_metadata", data) if isinstance(data, dict) else {}
    if not isinstance(metadata, dict):
        return objects

    for entry in metadata.values():
        if not isinstance(entry, dict):
            continue

        regions = entry.get("regions", [])
        if isinstance(regions, dict):
            if "shape_attributes" in regions:
                regions = [regions]
            else:
                regions = list(regions.values())

        if not isinstance(regions, list):
            continue

        for region in regions:
            shape = region.get("shape_attributes", {}) or {}
            attributes = region.get("region_attributes", {}) or {}

            if shape.get("name") != "rect":
                continue

            values = [
                shape.get("x"),
                shape.get("y"),
                shape.get("width"),
                shape.get("height"),
            ]

            if any(value is None for value in values):
                continue

            label = extract_via_label(attributes, label_aliases)
            if not label:
                continue

            try:
                x, y, width, height = [float(value) for value in values]
            except (TypeError, ValueError):
                continue

            if width > 1 and height > 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x, y, x + width, y + height],
                })

    return objects


def clip_box(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Optional[List[float]]:
    """Clip one xyxy box to image boundaries."""
    x1, y1, x2, y2 = box

    x1 = max(0.0, min(float(x1), image_width - 1.0))
    y1 = max(0.0, min(float(y1), image_height - 1.0))
    x2 = max(0.0, min(float(x2), float(image_width)))
    y2 = max(0.0, min(float(y2), float(image_height)))

    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None

    return [x1, y1, x2, y2]


## Module 3: Build, audit, and visually inspect the dataset

This module scans the selected year, keeps only the eight target weed classes, removes invalid boxes, saves an audit summary, plots class counts, and displays a small annotation preview for a visual check.


In [ ]:
def build_clean_records(
    source_root: Path,
    class_names: List[str],
    label_aliases: Dict[str, str],
    image_extensions: set,
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Build clean detection records from one dataset folder."""
    source_root = Path(source_root)

    if not source_root.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {source_root.resolve()}\n"
            "Update MAIN_DIRECTORY or DATASET_YEAR in Module 1."
        )

    image_root = next(
        (
            path
            for path in [
                source_root / "JPEGImages",
                source_root / "images",
                source_root,
            ]
            if path.exists()
        ),
        None,
    )

    if image_root is None:
        raise FileNotFoundError(f"No image root found below: {source_root.resolve()}")

    image_paths = sorted(
        path
        for path in image_root.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )

    if not image_paths:
        raise FileNotFoundError(f"No images found below: {image_root.resolve()}")

    image_names = [path.name for path in image_paths]
    if len(image_names) != len(set(image_names)):
        raise ValueError(
            "Duplicate image filenames were found. "
            "Flat YOLO split folders require unique filenames within one dataset."
        )

    xml_by_stem = {path.stem: path for path in source_root.rglob("*.xml")}
    json_by_stem = {path.stem: path for path in source_root.rglob("*.json")}
    target_class_set = set(class_names)

    records: List[Dict[str, Any]] = []
    stats = {
        "images_found": len(image_paths),
        "records_kept": 0,
        "missing_annotation": 0,
        "empty_after_filtering": 0,
        "unreadable_images": 0,
        "invalid_boxes_removed": 0,
        "non_target_objects_removed": 0,
        "used_xml": 0,
        "used_json": 0,
    }

    for image_path in image_paths:
        xml_path = xml_by_stem.get(image_path.stem)
        json_path = json_by_stem.get(image_path.stem)

        annotation_size = None
        objects: List[Dict[str, Any]] = []
        source_used = None

        try:
            if xml_path is not None:
                annotation_size, xml_objects = parse_pascal_voc_xml(
                    xml_path=xml_path,
                    label_aliases=label_aliases,
                )
                if xml_objects:
                    objects = xml_objects
                    source_used = "xml"

            if not objects and json_path is not None:
                json_objects = parse_json_annotation(
                    json_path=json_path,
                    label_aliases=label_aliases,
                )
                if json_objects:
                    objects = json_objects
                    source_used = "json"
                    annotation_size = None

            if not objects:
                stats["missing_annotation"] += 1
                continue

            if annotation_size is None:
                width, height = read_image_size(image_path)
            else:
                width, height = annotation_size

        except Exception as error:
            stats["unreadable_images"] += 1
            print(f"Skipping {image_path.name}: {error}")
            continue

        clean_objects: List[Dict[str, Any]] = []

        for obj in objects:
            label = normalize_label(obj["label"], label_aliases)

            if label not in target_class_set:
                stats["non_target_objects_removed"] += 1
                continue

            clipped = clip_box(
                box=obj["bbox_xyxy"],
                image_width=width,
                image_height=height,
            )

            if clipped is None:
                stats["invalid_boxes_removed"] += 1
                continue

            clean_objects.append({
                "label": label,
                "bbox_xyxy": clipped,
            })

        if not clean_objects:
            stats["empty_after_filtering"] += 1
            continue

        stats[f"used_{source_used}"] += 1

        records.append({
            "img_path": str(image_path.resolve()),
            "file_name": image_path.name,
            "width": int(width),
            "height": int(height),
            "objects": clean_objects,
        })

    records.sort(key=lambda row: row["file_name"])
    stats["records_kept"] = len(records)

    if not records:
        raise RuntimeError("No usable records remained after annotation filtering.")

    return records, stats


def audit_records(
    records_to_audit: List[Dict[str, Any]],
    class_names: List[str],
) -> Dict[str, Any]:
    """Count images, boxes, and class occurrences."""
    box_counts = {name: 0 for name in class_names}
    image_counts = {name: 0 for name in class_names}
    boxes_per_image: List[int] = []

    for record in records_to_audit:
        labels = [obj["label"] for obj in record["objects"]]
        boxes_per_image.append(len(labels))

        for label in labels:
            box_counts[label] += 1

        for label in set(labels):
            image_counts[label] += 1

    return {
        "num_images": len(records_to_audit),
        "num_boxes": int(sum(box_counts.values())),
        "box_count_per_class": box_counts,
        "image_count_per_class": image_counts,
        "missing_classes": [
            name for name in class_names if box_counts[name] == 0
        ],
        "boxes_per_image_mean": (
            float(mean(boxes_per_image)) if boxes_per_image else 0.0
        ),
    }


def plot_class_box_counts(
    audit: Dict[str, Any],
    class_names: List[str],
    output_path: Path,
) -> None:
    """Plot full dataset box counts by class."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    values = [audit["box_count_per_class"][name] for name in class_names]

    plt.figure(figsize=(10, 5))
    plt.bar(class_names, values)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Annotated boxes")
    plt.title("Full dataset class distribution")
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()


def plot_annotation_preview(
    records_to_plot: List[Dict[str, Any]],
    class_names: List[str],
    output_path: Path,
    sample_count: int = 4,
    seed: int = 2026,
) -> None:
    """Display a small reproducible sample with ground truth boxes."""
    if not records_to_plot:
        return

    output_path.parent.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    selected = rng.sample(
        records_to_plot,
        k=min(sample_count, len(records_to_plot)),
    )

    columns = 2
    rows = math.ceil(len(selected) / columns)

    fig, axes = plt.subplots(rows, columns, figsize=(12, 6 * rows))
    axes = np.atleast_1d(axes).reshape(-1)

    class_to_id = {name: idx for idx, name in enumerate(class_names)}

    for axis, record in zip(axes, selected):
        with Image.open(record["img_path"]) as image:
            image_rgb = image.convert("RGB")
            draw = ImageDraw.Draw(image_rgb)

            for obj in record["objects"]:
                x1, y1, x2, y2 = obj["bbox_xyxy"]
                draw.rectangle([x1, y1, x2, y2], width=3)
                label = obj["label"]
                draw.text((x1 + 3, y1 + 3), f"{class_to_id[label]}:{label}")

            axis.imshow(image_rgb)

        axis.set_title(record["file_name"])
        axis.axis("off")

    for axis in axes[len(selected):]:
        axis.axis("off")

    plt.suptitle("Annotation sanity check")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()


records, DATASET_SCAN_STATS = build_clean_records(
    source_root=SOURCE_DATASET_ROOT,
    class_names=CLASS_NAMES,
    label_aliases=LABEL_ALIASES,
    image_extensions=IMAGE_EXTENSIONS,
)

DATASET_AUDIT = audit_records(
    records_to_audit=records,
    class_names=CLASS_NAMES,
)

if DATASET_AUDIT["missing_classes"]:
    raise RuntimeError(
        f"Dataset is missing target classes: {DATASET_AUDIT['missing_classes']}"
    )

(STATS_ROOT / f"{DATASET_TAG}_scan_summary.json").write_text(
    json.dumps(DATASET_SCAN_STATS, indent=2),
    encoding="utf-8",
)

(STATS_ROOT / f"{DATASET_TAG}_class_audit.json").write_text(
    json.dumps(DATASET_AUDIT, indent=2),
    encoding="utf-8",
)

print("\nDATASET SCAN")
print(json.dumps(DATASET_SCAN_STATS, indent=2))

print("\nDATASET AUDIT")
print(json.dumps(DATASET_AUDIT, indent=2))

plot_class_box_counts(
    audit=DATASET_AUDIT,
    class_names=CLASS_NAMES,
    output_path=RUN_PLOTS_ROOT / "full_dataset_class_distribution.png",
)

plot_annotation_preview(
    records_to_plot=records,
    class_names=CLASS_NAMES,
    output_path=RUN_PLOTS_ROOT / "annotation_sanity_check.png",
)


## Module 4: Deterministic 70/15/15 splitting and leakage checks

This module creates reproducible train, validation, and test splits from each seed. It checks that no image appears in more than one split and verifies class coverage before training.


In [ ]:
def split_records_deterministically(
    all_records: List[Dict[str, Any]],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    seed: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Create one reproducible 70/15/15 split."""
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0.")

    shuffled = list(all_records)
    rng = random.Random(seed)
    rng.shuffle(shuffled)

    total = len(shuffled)
    train_count = int(total * train_ratio)
    valid_count = int(total * valid_ratio)

    split_records = {
        "train": shuffled[:train_count],
        "val": shuffled[train_count:train_count + valid_count],
        "test": shuffled[train_count + valid_count:],
    }

    if sum(len(rows) for rows in split_records.values()) != total:
        raise RuntimeError("Split count integrity check failed.")

    return split_records


def verify_split_disjointness(
    split_records: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Verify that filenames do not overlap across train, validation, and test."""
    file_sets = {
        split_name: {row["file_name"] for row in rows}
        for split_name, rows in split_records.items()
    }

    pairs = [("train", "val"), ("train", "test"), ("val", "test")]

    for left, right in pairs:
        overlap = file_sets[left].intersection(file_sets[right])
        if overlap:
            raise RuntimeError(
                f"Data leakage detected between {left} and {right}: "
                f"{sorted(overlap)[:10]}"
            )


def audit_split_coverage(
    split_records: Dict[str, List[Dict[str, Any]]],
    class_names: List[str],
    require_all_classes: bool,
) -> Dict[str, Dict[str, Any]]:
    """Audit every split and optionally require every class in every split."""
    coverage = {
        split_name: audit_records(rows, class_names)
        for split_name, rows in split_records.items()
    }

    if require_all_classes:
        failures = {
            split_name: audit["missing_classes"]
            for split_name, audit in coverage.items()
            if audit["missing_classes"]
        }

        if failures:
            raise RuntimeError(
                "One or more splits are missing target classes. "
                f"Coverage failure: {failures}"
            )

    return coverage


def save_split_manifest(
    split_records: Dict[str, List[Dict[str, Any]]],
    output_path: Path,
    seed: int,
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    dataset_tag: str,
) -> None:
    """Save the exact file assignment for reproducibility."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    manifest = {
        "dataset_tag": dataset_tag,
        "seed": int(seed),
        "ratios": {
            "train": float(train_ratio),
            "val": float(valid_ratio),
            "test": float(test_ratio),
        },
        "counts": {
            split_name: len(rows)
            for split_name, rows in split_records.items()
        },
        "files": {
            split_name: [row["file_name"] for row in rows]
            for split_name, rows in split_records.items()
        },
    }

    output_path.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )


PREVIEW_SPLITS = split_records_deterministically(
    all_records=records,
    train_ratio=TRAIN_RATIO,
    valid_ratio=VALID_RATIO,
    test_ratio=TEST_RATIO,
    seed=FINAL_EVALUATION_SEEDS[0],
)

verify_split_disjointness(PREVIEW_SPLITS)

PREVIEW_COVERAGE = audit_split_coverage(
    split_records=PREVIEW_SPLITS,
    class_names=CLASS_NAMES,
    require_all_classes=REQUIRE_ALL_CLASSES_PER_SPLIT,
)

print("\nPREVIEW SPLIT USING SEED", FINAL_EVALUATION_SEEDS[0])
for split_name, audit in PREVIEW_COVERAGE.items():
    print(
        f"{split_name:5s} images={audit['num_images']:5d} "
        f"boxes={audit['num_boxes']:5d} "
        f"missing={audit['missing_classes']}"
    )


## Module 5: Export one split to YOLO format

This module creates temporary YOLO image and label folders for each seed. It uses symlinks when possible, falls back to hardlinks or copies when needed, writes `dataset.yaml`, and checks image and label counts before training.


In [ ]:
def remove_file_or_link(path: Path) -> None:
    """Remove a regular file or symlink, including a broken symlink."""
    if path.is_symlink() or path.is_file():
        path.unlink()


def clear_split_directory(directory: Path) -> None:
    """Remove files and symlinks from one temporary split folder."""
    directory.mkdir(parents=True, exist_ok=True)

    for path in directory.iterdir():
        if path.is_file() or path.is_symlink():
            path.unlink()


def place_image(source: Path, destination: Path, mode: str) -> str:
    """Place one image with a safe fallback sequence."""
    if mode not in {"symlink", "hardlink", "copy"}:
        raise ValueError("IMAGE_PLACEMENT_MODE must be symlink, hardlink, or copy.")

    source = source.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    remove_file_or_link(destination)

    if mode == "symlink":
        try:
            destination.symlink_to(source)
            return "symlinked"
        except OSError:
            pass

    if mode in {"symlink", "hardlink"}:
        try:
            os.link(source, destination)
            return "hardlinked"
        except OSError:
            pass

    shutil.copy2(source, destination)
    return "copied"


def xyxy_to_yolo(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Tuple[float, float, float, float]:
    """Convert absolute xyxy coordinates to normalized YOLO xywh."""
    x1, y1, x2, y2 = box

    values = (
        ((x1 + x2) / 2.0) / image_width,
        ((y1 + y2) / 2.0) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )

    return tuple(
        max(0.0, min(1.0, float(value)))
        for value in values
    )


def write_yolo_label(
    record: Dict[str, Any],
    output_path: Path,
    class_to_id: Dict[str, int],
) -> None:
    """Write one YOLO label file."""
    lines: List[str] = []

    for obj in record["objects"]:
        class_id = class_to_id[obj["label"]]
        xc, yc, width, height = xyxy_to_yolo(
            box=obj["bbox_xyxy"],
            image_width=record["width"],
            image_height=record["height"],
        )

        lines.append(
            f"{class_id} {xc:.6f} {yc:.6f} {width:.6f} {height:.6f}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )


def write_dataset_yaml(
    dataset_root: Path,
    class_names: List[str],
) -> Path:
    """Write an Ultralytics dataset configuration file."""
    yaml_path = dataset_root / "dataset.yaml"

    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/val",
        "test: images/test",
        f"nc: {len(class_names)}",
        "names:",
    ]

    lines.extend(
        f"  {index}: {name}"
        for index, name in enumerate(class_names)
    )

    yaml_path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )

    return yaml_path


def count_image_files(
    directory: Path,
    image_extensions: set,
) -> int:
    """Count only image files and image symlinks."""
    return sum(
        1
        for path in directory.iterdir()
        if (path.is_file() or path.is_symlink())
        and path.suffix.lower() in image_extensions
    )


def count_label_files(directory: Path) -> int:
    """Count YOLO text label files."""
    return sum(
        1
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".txt"
    )


def check_yolo_dataset_integrity(
    dataset_root: Path,
    expected_counts: Dict[str, int],
    image_extensions: set,
) -> None:
    """Verify image and label counts for every split."""
    for split_name in ("train", "val", "test"):
        image_dir = dataset_root / "images" / split_name
        label_dir = dataset_root / "labels" / split_name

        image_count = count_image_files(
            directory=image_dir,
            image_extensions=image_extensions,
        )
        label_count = count_label_files(label_dir)
        expected = int(expected_counts[split_name])

        print(
            f"{split_name:5s} images={image_count:5d} "
            f"labels={label_count:5d} expected={expected:5d}"
        )

        if image_count != expected or label_count != expected:
            raise RuntimeError(
                f"YOLO integrity failure in {split_name}: "
                f"images={image_count}, labels={label_count}, expected={expected}."
            )


def prepare_yolo_dataset(
    split_records: Dict[str, List[Dict[str, Any]]],
    output_root: Path,
    manifest_root: Path,
    image_mode: str,
    seed: int,
    class_names: List[str],
    image_extensions: set,
    dataset_tag: str,
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    require_all_classes: bool,
) -> Tuple[Path, Dict[str, Dict[str, Any]]]:
    """Build and verify one temporary YOLO dataset for one seed."""
    output_root = Path(output_root)
    manifest_root = Path(manifest_root)

    verify_split_disjointness(split_records)

    coverage = audit_split_coverage(
        split_records=split_records,
        class_names=class_names,
        require_all_classes=require_all_classes,
    )

    class_to_id = {
        name: index
        for index, name in enumerate(class_names)
    }

    image_dirs = {
        split_name: output_root / "images" / split_name
        for split_name in ("train", "val", "test")
    }

    label_dirs = {
        split_name: output_root / "labels" / split_name
        for split_name in ("train", "val", "test")
    }

    for directory in [*image_dirs.values(), *label_dirs.values()]:
        clear_split_directory(directory)

    placement_counts = {
        "symlinked": 0,
        "hardlinked": 0,
        "copied": 0,
    }

    for split_name, rows in split_records.items():
        for record in rows:
            source = Path(record["img_path"])
            destination = image_dirs[split_name] / record["file_name"]

            action = place_image(
                source=source,
                destination=destination,
                mode=image_mode,
            )
            placement_counts[action] += 1

            label_path = (
                label_dirs[split_name]
                / f"{Path(record['file_name']).stem}.txt"
            )

            write_yolo_label(
                record=record,
                output_path=label_path,
                class_to_id=class_to_id,
            )

    dataset_yaml = write_dataset_yaml(
        dataset_root=output_root,
        class_names=class_names,
    )

    save_split_manifest(
        split_records=split_records,
        output_path=manifest_root / f"{dataset_tag}_seed{seed}_split.json",
        seed=seed,
        train_ratio=train_ratio,
        valid_ratio=valid_ratio,
        test_ratio=test_ratio,
        dataset_tag=dataset_tag,
    )

    expected_counts = {
        split_name: len(rows)
        for split_name, rows in split_records.items()
    }

    check_yolo_dataset_integrity(
        dataset_root=output_root,
        expected_counts=expected_counts,
        image_extensions=image_extensions,
    )

    layout_manifest = {
        "dataset_tag": dataset_tag,
        "seed": int(seed),
        "dataset_yaml": str(dataset_yaml.resolve()),
        "temporary_yolo_root": str(output_root.resolve()),
        "placement_counts": placement_counts,
        "expected_counts": expected_counts,
        "coverage": coverage,
    }

    manifest_root.mkdir(parents=True, exist_ok=True)
    (manifest_root / "yolo_layout_manifest.json").write_text(
        json.dumps(layout_manifest, indent=2),
        encoding="utf-8",
    )

    print("Dataset YAML:", dataset_yaml.resolve())
    print("Image placement:", placement_counts)

    return dataset_yaml, coverage


## Module 6: Metric extraction, training, validation, and isolated test evaluation

This module trains one untuned YOLOv8m model from the pretrained checkpoint, verifies the saved best checkpoint, re-evaluates `best.pt` on validation, and evaluates the frozen checkpoint on test only after training has finished.


In [ ]:
def extract_box_metrics(
    metrics: Any,
    expected_class_names: List[str],
) -> Dict[str, Any]:
    """Extract overall and per class detection metrics."""
    box = getattr(metrics, "box", None)

    if box is None:
        raise RuntimeError("Ultralytics returned no box metrics.")

    precision = float(box.mp)
    recall = float(box.mr)
    map50 = float(box.map50)
    map50_95 = float(box.map)

    names_obj = getattr(metrics, "names", {})

    if isinstance(names_obj, dict):
        class_name_lookup = {
            int(key): str(value)
            for key, value in names_obj.items()
        }
    else:
        class_name_lookup = {
            index: str(name)
            for index, name in enumerate(names_obj)
        }

    per_class_rows: List[Dict[str, Any]] = []

    if hasattr(box, "ap") and hasattr(box, "ap_class_index"):
        ap_values = np.asarray(box.ap, dtype=float).reshape(-1)
        class_ids = np.asarray(box.ap_class_index, dtype=int).reshape(-1)

        for class_id, ap_value in zip(
            class_ids.tolist(),
            ap_values.tolist(),
        ):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(
                    int(class_id),
                    str(class_id),
                ),
                "map50_95": float(ap_value),
            })
    else:
        maps = [float(value) for value in box.maps]

        for class_id, ap_value in enumerate(maps):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(
                    int(class_id),
                    str(class_id),
                ),
                "map50_95": float(ap_value),
            })

    represented_ids = {
        int(row["class_id"])
        for row in per_class_rows
    }
    expected_ids = set(range(len(expected_class_names)))
    missing_ids = sorted(expected_ids - represented_ids)

    if missing_ids:
        missing_names = [
            expected_class_names[index]
            for index in missing_ids
        ]
        raise RuntimeError(
            "Evaluation metrics are missing represented classes: "
            f"{missing_names}"
        )

    sorted_aps = sorted(
        float(row["map50_95"])
        for row in per_class_rows
    )

    bottom3 = float(mean(sorted_aps[:3]))

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "bottom3_mean_map50_95": bottom3,
        "per_class_map50_95": per_class_rows,
    }


def read_epoch_map50_95_values(
    results_csv: Path,
) -> Tuple[str, List[float]]:
    """Read epoch level validation mAP50-95 from results.csv."""
    results_csv = Path(results_csv)

    if not results_csv.exists():
        raise FileNotFoundError(
            f"Training results.csv not found: {results_csv}"
        )

    with results_csv.open(
        "r",
        newline="",
        encoding="utf-8-sig",
    ) as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    candidate_columns = [
        name
        for name in fieldnames
        if "mAP50-95" in name
    ]

    if not candidate_columns:
        raise RuntimeError(
            "Could not find an mAP50-95 column in results.csv. "
            f"Available columns: {fieldnames}"
        )

    metric_column = next(
        (
            name
            for name in candidate_columns
            if "metrics/" in name
        ),
        candidate_columns[0],
    )

    values: List[float] = []

    for row in rows:
        raw_value = row.get(metric_column, "")

        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            continue

        if math.isfinite(value):
            values.append(value)

    if not values:
        raise RuntimeError(
            f"No numeric values were found in column: {metric_column}"
        )

    return metric_column, values


def verify_ultralytics_fitness_alignment(
    results_csv: Path,
    trainer_best_fitness: Optional[float],
    tolerance: float,
    strict: bool,
) -> Dict[str, Any]:
    """Check that trainer best fitness matches the maximum epoch mAP50-95."""
    metric_column, map95_values = read_epoch_map50_95_values(results_csv)
    max_epoch_map95 = float(max(map95_values))

    if (
        trainer_best_fitness is None
        or not math.isfinite(float(trainer_best_fitness))
    ):
        report = {
            "status": "UNVERIFIED",
            "metric_column": metric_column,
            "max_epoch_map50_95": max_epoch_map95,
            "trainer_best_fitness": None,
            "absolute_difference": None,
            "tolerance": float(tolerance),
        }

        if strict:
            raise RuntimeError(
                "trainer.best_fitness was unavailable, so checkpoint alignment "
                "could not be verified."
            )

        return report

    trainer_best_fitness = float(trainer_best_fitness)
    difference = abs(trainer_best_fitness - max_epoch_map95)
    status = "PASS" if difference <= tolerance else "FAIL"

    report = {
        "status": status,
        "metric_column": metric_column,
        "max_epoch_map50_95": max_epoch_map95,
        "trainer_best_fitness": trainer_best_fitness,
        "absolute_difference": float(difference),
        "tolerance": float(tolerance),
    }

    if strict and status != "PASS":
        raise RuntimeError(
            "Ultralytics checkpoint fitness does not align with mAP50-95. "
            f"best_fitness={trainer_best_fitness:.6f}, "
            f"max_epoch_map50_95={max_epoch_map95:.6f}, "
            f"difference={difference:.6f}"
        )

    return report


def resolve_device() -> str:
    """Use GPU device 0 when CUDA is available, otherwise use CPU."""
    return "0" if torch.cuda.is_available() else "cpu"


def train_untuned_yolov8m(
    dataset_yaml: Path,
    project_dir: Path,
    model_name: str,
    dataset_tag: str,
    epochs: int,
    image_size: int,
    batch_size: int,
    patience: int,
    workers: int,
    seed: int,
    warmup_epochs: float,
    close_mosaic: int,
    untuned_hyperparameters: Dict[str, float],
    fixed_training_args: Dict[str, Any],
    expected_class_names: List[str],
    strict_fitness_alignment_check: bool,
    fitness_alignment_tolerance: float,
    device: str,
) -> Dict[str, Any]:
    """Train one full fidelity untuned YOLOv8m run."""
    dataset_yaml = Path(dataset_yaml)
    project_dir = Path(project_dir)

    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")

    if epochs < 1 or batch_size < 1 or workers < 0:
        raise ValueError("Invalid epochs, batch size, or workers value.")

    if warmup_epochs < 0 or close_mosaic < 0:
        raise ValueError(
            "warmup_epochs and close_mosaic must be nonnegative."
        )

    required_hyperparameters = {
        "lr0",
        "lrf",
        "momentum",
        "weight_decay",
        "scale",
    }

    missing = required_hyperparameters.difference(
        untuned_hyperparameters
    )

    if missing:
        raise KeyError(
            f"Untuned hyperparameter dictionary is missing: {sorted(missing)}"
        )

    using_cuda = device != "cpu" and torch.cuda.is_available()

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    trial_id = uuid.uuid4().hex[:8]
    run_name = (
        f"{dataset_tag}_untuned_yolov8m_seed{seed}_"
        f"e{epochs}_{trial_id}"
    )

    project_dir.mkdir(parents=True, exist_ok=True)

    train_kwargs = dict(fixed_training_args)

    train_kwargs.update({
        "data": str(dataset_yaml.resolve()),
        "project": str(project_dir.resolve()),
        "name": run_name,
        "epochs": int(epochs),
        "imgsz": int(image_size),
        "batch": int(batch_size),
        "patience": int(patience),
        "workers": int(workers),
        "device": device,
        "seed": int(seed),
        "deterministic": True,
        "pretrained": True,
        "val": True,
        "save": True,
        "save_period": -1,
        "plots": False,
        "verbose": True,
        "amp": bool(using_cuda),
        "lr0": float(untuned_hyperparameters["lr0"]),
        "lrf": float(untuned_hyperparameters["lrf"]),
        "momentum": float(untuned_hyperparameters["momentum"]),
        "weight_decay": float(untuned_hyperparameters["weight_decay"]),
        "scale": float(untuned_hyperparameters["scale"]),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
    })

    print("\n" + "=" * 92)
    print("START UNTUNED TRAINING")
    print("Run:", run_name)
    print("Seed:", seed)
    print("Device:", device)
    print("Epochs:", epochs)
    print("Untuned hyperparameters:", json.dumps(
        untuned_hyperparameters,
        indent=2,
    ))
    print("=" * 92)

    model = YOLO(model_name)

    training_start = time.perf_counter()
    train_result = model.train(**train_kwargs)
    training_seconds = time.perf_counter() - training_start

    train_save_dir = getattr(train_result, "save_dir", None)

    if train_save_dir is None and getattr(model, "trainer", None) is not None:
        train_save_dir = getattr(model.trainer, "save_dir", None)

    if train_save_dir is None:
        raise RuntimeError(
            "Training finished but the Ultralytics save directory was unavailable."
        )

    train_save_dir = Path(train_save_dir)
    best_checkpoint = train_save_dir / "weights" / "best.pt"
    results_csv = train_save_dir / "results.csv"

    if not best_checkpoint.exists():
        raise FileNotFoundError(f"best.pt not found: {best_checkpoint}")

    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv}")

    trainer_best_fitness = None
    trainer = getattr(model, "trainer", None)

    if trainer is not None:
        raw_best_fitness = getattr(trainer, "best_fitness", None)

        if raw_best_fitness is not None:
            trainer_best_fitness = float(raw_best_fitness)

    alignment_report = verify_ultralytics_fitness_alignment(
        results_csv=results_csv,
        trainer_best_fitness=trainer_best_fitness,
        tolerance=fitness_alignment_tolerance,
        strict=strict_fitness_alignment_check,
    )

    best_model = YOLO(str(best_checkpoint))

    validation_start = time.perf_counter()
    validation_metrics = best_model.val(
        data=str(dataset_yaml.resolve()),
        split="val",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(project_dir.resolve()),
        name=f"{run_name}_validation",
        plots=False,
        save_json=False,
        verbose=False,
    )
    validation_seconds = time.perf_counter() - validation_start

    validation_data = extract_box_metrics(
        metrics=validation_metrics,
        expected_class_names=expected_class_names,
    )

    peak_gpu_memory_gb = None

    if using_cuda:
        peak_gpu_memory_gb = (
            torch.cuda.max_memory_allocated() / (1024 ** 3)
        )

    result = {
        "status": "PASS",
        "trial_id": trial_id,
        "run_name": run_name,
        "seed": int(seed),
        "untuned_hyperparameters": dict(untuned_hyperparameters),
        "validation_metrics": validation_data,
        "fitness_alignment": alignment_report,
        "timing": {
            "training_seconds": float(training_seconds),
            "validation_seconds": float(validation_seconds),
        },
        "hardware": {
            "device": str(device),
            "gpu_name": (
                torch.cuda.get_device_name(0)
                if using_cuda
                else None
            ),
            "peak_gpu_memory_gb": (
                float(peak_gpu_memory_gb)
                if peak_gpu_memory_gb is not None
                else None
            ),
        },
        "paths": {
            "training_folder": str(train_save_dir.resolve()),
            "best_checkpoint": str(best_checkpoint.resolve()),
            "results_csv": str(results_csv.resolve()),
        },
    }

    (train_save_dir / "trial_summary.json").write_text(
        json.dumps(result, indent=2),
        encoding="utf-8",
    )

    print(
        f"Completed seed {seed}: "
        f"validation mAP50-95={validation_data['map50_95']:.6f}"
    )

    return result


def evaluate_checkpoint_on_test(
    checkpoint_path: Path,
    dataset_yaml: Path,
    output_root: Path,
    run_name: str,
    expected_class_names: List[str],
    image_size: int,
    batch_size: int,
    workers: int,
    device: str,
) -> Dict[str, Any]:
    """Evaluate one frozen checkpoint on the isolated test split."""
    checkpoint_path = Path(checkpoint_path)
    dataset_yaml = Path(dataset_yaml)
    output_root = Path(output_root)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    if not dataset_yaml.exists():
        raise FileNotFoundError(
            f"Dataset YAML not found: {dataset_yaml}"
        )

    output_root.mkdir(parents=True, exist_ok=True)

    model = YOLO(str(checkpoint_path))

    test_start = time.perf_counter()
    test_metrics = model.val(
        data=str(dataset_yaml.resolve()),
        split="test",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(output_root.resolve()),
        name=run_name,
        plots=False,
        save_json=False,
        verbose=False,
    )
    test_seconds = time.perf_counter() - test_start

    metric_data = extract_box_metrics(
        metrics=test_metrics,
        expected_class_names=expected_class_names,
    )

    return {
        "precision": metric_data["precision"],
        "recall": metric_data["recall"],
        "map50": metric_data["map50"],
        "map50_95": metric_data["map50_95"],
        "bottom3_mean_map50_95": (
            metric_data["bottom3_mean_map50_95"]
        ),
        "test_seconds": float(test_seconds),
        "per_class_map50_95": metric_data["per_class_map50_95"],
    }


## Module 7: Result saving and diagnostic plots

This module saves tabular results and creates two compact diagnostics after all three seeds finish: the validation mAP50-95 training trajectories and the mean test metrics with standard deviation.


In [ ]:
def save_rows_to_csv(
    rows: List[Dict[str, Any]],
    output_path: Path,
) -> None:
    """Save a list of dictionaries with a stable union of field names."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if not rows:
        output_path.write_text("", encoding="utf-8")
        return

    fieldnames: List[str] = []

    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)

    with output_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
        )
        writer.writeheader()
        writer.writerows(rows)


def summarize_numeric_rows(
    rows: List[Dict[str, Any]],
    metric_names: List[str],
) -> List[Dict[str, Any]]:
    """Summarize selected numeric fields with mean and sample standard deviation."""
    summary_rows: List[Dict[str, Any]] = []

    for metric_name in metric_names:
        values = [
            float(row[metric_name])
            for row in rows
            if row.get(metric_name) not in {None, ""}
        ]

        if not values:
            continue

        summary_rows.append({
            "metric": metric_name,
            "n": len(values),
            "mean": float(mean(values)),
            "sd": (
                float(stdev(values))
                if len(values) > 1
                else 0.0
            ),
            "min": float(min(values)),
            "max": float(max(values)),
        })

    return summary_rows


def plot_training_trajectories(
    final_rows: List[Dict[str, Any]],
    output_path: Path,
) -> None:
    """Plot validation mAP50-95 across epochs for all final seeds."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(9, 5))

    plotted = 0

    for row in final_rows:
        results_csv = Path(row["results_csv"])

        if not results_csv.exists():
            continue

        _, values = read_epoch_map50_95_values(results_csv)

        epochs = list(range(1, len(values) + 1))
        plt.plot(
            epochs,
            values,
            label=f"Seed {row['seed']}",
        )
        plotted += 1

    if plotted == 0:
        plt.close()
        return

    plt.xlabel("Epoch")
    plt.ylabel("Validation mAP50-95")
    plt.title("Untuned YOLOv8m training trajectories")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()


def plot_test_metric_summary(
    final_rows: List[Dict[str, Any]],
    output_path: Path,
) -> None:
    """Plot mean test metrics with sample standard deviation."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    metric_keys = [
        "test_precision",
        "test_recall",
        "test_map50",
        "test_map50_95",
    ]

    labels = [
        "Precision",
        "Recall",
        "mAP50",
        "mAP50-95",
    ]

    means = [
        mean(float(row[key]) for row in final_rows)
        for key in metric_keys
    ]

    standard_deviations = [
        stdev(float(row[key]) for row in final_rows)
        if len(final_rows) > 1
        else 0.0
        for key in metric_keys
    ]

    plt.figure(figsize=(8, 5))
    plt.bar(
        labels,
        means,
        yerr=standard_deviations,
        capsize=5,
    )
    plt.ylim(0.0, 1.0)
    plt.ylabel("Metric value")
    plt.title("Untuned YOLOv8m test performance: mean and SD")
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()


## Module 8: Run one complete seed

This module performs the complete workflow for one seed: split the records, export temporary YOLO data, train the untuned YOLOv8m model, freeze `best.pt`, evaluate the test split, and return one reproducible result row.


In [ ]:
def run_single_seed(
    seed: int,
    all_records: List[Dict[str, Any]],
    dataset_tag: str,
    class_names: List[str],
    image_extensions: set,
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    require_all_classes: bool,
    yolo_data_root: Path,
    splits_root: Path,
    training_root: Path,
    test_eval_root: Path,
    image_mode: str,
    model_name: str,
    epochs: int,
    image_size: int,
    batch_size: int,
    workers: int,
    patience: int,
    warmup_epochs: float,
    close_mosaic: int,
    untuned_hyperparameters: Dict[str, float],
    fixed_training_args: Dict[str, Any],
    strict_fitness_alignment_check: bool,
    fitness_alignment_tolerance: float,
    device: str,
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """Run one complete untuned baseline replication."""
    print("\n" + "=" * 92)
    print(f"UNTUNED YOLOv8m FINAL RUN: SEED {seed}")
    print("=" * 92)

    split_records = split_records_deterministically(
        all_records=all_records,
        train_ratio=train_ratio,
        valid_ratio=valid_ratio,
        test_ratio=test_ratio,
        seed=seed,
    )

    temporary_dataset_root = (
        Path(yolo_data_root)
        / f"{dataset_tag}_seed{seed}"
    )

    manifest_root = (
        Path(splits_root)
        / dataset_tag
        / f"seed{seed}"
    )

    dataset_yaml, coverage = prepare_yolo_dataset(
        split_records=split_records,
        output_root=temporary_dataset_root,
        manifest_root=manifest_root,
        image_mode=image_mode,
        seed=seed,
        class_names=class_names,
        image_extensions=image_extensions,
        dataset_tag=dataset_tag,
        train_ratio=train_ratio,
        valid_ratio=valid_ratio,
        test_ratio=test_ratio,
        require_all_classes=require_all_classes,
    )

    run_start = time.perf_counter()

    training_result = train_untuned_yolov8m(
        dataset_yaml=dataset_yaml,
        project_dir=training_root,
        model_name=model_name,
        dataset_tag=dataset_tag,
        epochs=epochs,
        image_size=image_size,
        batch_size=batch_size,
        patience=patience,
        workers=workers,
        seed=seed,
        warmup_epochs=warmup_epochs,
        close_mosaic=close_mosaic,
        untuned_hyperparameters=untuned_hyperparameters,
        fixed_training_args=fixed_training_args,
        expected_class_names=class_names,
        strict_fitness_alignment_check=(
            strict_fitness_alignment_check
        ),
        fitness_alignment_tolerance=(
            fitness_alignment_tolerance
        ),
        device=device,
    )

    test_result = evaluate_checkpoint_on_test(
        checkpoint_path=Path(
            training_result["paths"]["best_checkpoint"]
        ),
        dataset_yaml=dataset_yaml,
        output_root=test_eval_root,
        run_name=f"{dataset_tag}_seed{seed}_test",
        expected_class_names=class_names,
        image_size=image_size,
        batch_size=batch_size,
        workers=workers,
        device=device,
    )

    run_wall_seconds = time.perf_counter() - run_start
    validation_metrics = training_result["validation_metrics"]

    final_row = {
        "dataset_tag": dataset_tag,
        "model_name": model_name,
        "seed": int(seed),
        "train_images": len(split_records["train"]),
        "val_images": len(split_records["val"]),
        "test_images": len(split_records["test"]),
        "epochs": int(epochs),
        "imgsz": int(image_size),
        "batch": int(batch_size),
        "workers": int(workers),
        "patience": int(patience),
        "optimizer": fixed_training_args["optimizer"],
        "cache": fixed_training_args["cache"],
        "lr0": float(untuned_hyperparameters["lr0"]),
        "lrf": float(untuned_hyperparameters["lrf"]),
        "momentum": float(untuned_hyperparameters["momentum"]),
        "weight_decay": float(
            untuned_hyperparameters["weight_decay"]
        ),
        "scale": float(untuned_hyperparameters["scale"]),
        "val_precision": float(validation_metrics["precision"]),
        "val_recall": float(validation_metrics["recall"]),
        "val_map50": float(validation_metrics["map50"]),
        "val_map50_95": float(
            validation_metrics["map50_95"]
        ),
        "val_bottom3_mean_map50_95": float(
            validation_metrics["bottom3_mean_map50_95"]
        ),
        "test_precision": float(test_result["precision"]),
        "test_recall": float(test_result["recall"]),
        "test_map50": float(test_result["map50"]),
        "test_map50_95": float(test_result["map50_95"]),
        "test_bottom3_mean_map50_95": float(
            test_result["bottom3_mean_map50_95"]
        ),
        "training_seconds": float(
            training_result["timing"]["training_seconds"]
        ),
        "validation_seconds": float(
            training_result["timing"]["validation_seconds"]
        ),
        "test_seconds": float(test_result["test_seconds"]),
        "run_wall_seconds": float(run_wall_seconds),
        "fitness_alignment_status": (
            training_result["fitness_alignment"]["status"]
        ),
        "best_checkpoint": (
            training_result["paths"]["best_checkpoint"]
        ),
        "results_csv": training_result["paths"]["results_csv"],
        "split_manifest": str(
            (
                manifest_root
                / f"{dataset_tag}_seed{seed}_split.json"
            ).resolve()
        ),
    }

    per_class_rows = [
        {
            "dataset_tag": dataset_tag,
            "seed": int(seed),
            "class_id": int(row["class_id"]),
            "class_name": row["class_name"],
            "test_map50_95": float(row["map50_95"]),
        }
        for row in test_result["per_class_map50_95"]
    ]

    seed_summary = {
        "final_row": final_row,
        "split_coverage": coverage,
        "per_class_test_metrics": per_class_rows,
    }

    (
        Path(manifest_root)
        / f"{dataset_tag}_seed{seed}_result.json"
    ).write_text(
        json.dumps(seed_summary, indent=2),
        encoding="utf-8",
    )

    print(
        f"Seed {seed} complete: "
        f"test mAP50-95={final_row['test_map50_95']:.6f}"
    )

    return final_row, per_class_rows


## Module 9: Run the three seed untuned baseline and summarize results

This module runs seeds 42, 123, and 999, saves results after every completed seed, calculates mean and standard deviation, summarizes per class test mAP50-95, and generates the final diagnostic plots.


In [ ]:
def run_untuned_yolov8m_baseline(
    all_records: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """Run the complete three seed untuned YOLOv8m baseline."""
    workflow_start = time.perf_counter()
    device = resolve_device()

    final_results_csv = (
        RUN_STATS_ROOT
        / "untuned_yolov8m_three_seed_results.csv"
    )
    final_summary_csv = (
        RUN_STATS_ROOT
        / "untuned_yolov8m_mean_sd_summary.csv"
    )
    per_class_csv = (
        RUN_STATS_ROOT
        / "untuned_yolov8m_test_per_class.csv"
    )
    per_class_summary_csv = (
        RUN_STATS_ROOT
        / "untuned_yolov8m_test_per_class_mean_sd.csv"
    )
    manifest_json = (
        RUN_STATS_ROOT
        / "untuned_yolov8m_experiment_manifest.json"
    )

    print("\n" + "=" * 92)
    print("START THREE SEED UNTUNED YOLOv8m BASELINE")
    print("=" * 92)
    print("Dataset:", DATASET_TAG)
    print("Device:", device)
    print("Seeds:", FINAL_EVALUATION_SEEDS)
    print("Epochs per run:", FULL_FIDELITY_EPOCHS)
    print("Total baseline epoch units:", BASELINE_EPOCH_UNITS)
    print("Workers:", WORKERS)
    print("Cache:", CACHE_MODE)
    print("Patience:", PATIENCE)
    print("=" * 92)

    final_rows: List[Dict[str, Any]] = []
    per_class_rows: List[Dict[str, Any]] = []

    for seed in FINAL_EVALUATION_SEEDS:
        final_row, seed_per_class_rows = run_single_seed(
            seed=int(seed),
            all_records=all_records,
            dataset_tag=DATASET_TAG,
            class_names=CLASS_NAMES,
            image_extensions=IMAGE_EXTENSIONS,
            train_ratio=TRAIN_RATIO,
            valid_ratio=VALID_RATIO,
            test_ratio=TEST_RATIO,
            require_all_classes=REQUIRE_ALL_CLASSES_PER_SPLIT,
            yolo_data_root=YOLO_DATA_ROOT,
            splits_root=SPLITS_ROOT,
            training_root=TRAINING_ROOT,
            test_eval_root=TEST_EVAL_ROOT,
            image_mode=IMAGE_PLACEMENT_MODE,
            model_name=MODEL_NAME,
            epochs=FULL_FIDELITY_EPOCHS,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            workers=WORKERS,
            patience=PATIENCE,
            warmup_epochs=FULL_FIDELITY_WARMUP_EPOCHS,
            close_mosaic=FULL_FIDELITY_CLOSE_MOSAIC,
            untuned_hyperparameters=UNTUNED_HYPERPARAMETERS,
            fixed_training_args=FIXED_TRAINING_ARGS,
            strict_fitness_alignment_check=(
                STRICT_FITNESS_ALIGNMENT_CHECK
            ),
            fitness_alignment_tolerance=(
                FITNESS_ALIGNMENT_TOLERANCE
            ),
            device=device,
        )

        final_rows.append(final_row)
        per_class_rows.extend(seed_per_class_rows)

        save_rows_to_csv(
            rows=final_rows,
            output_path=final_results_csv,
        )

        save_rows_to_csv(
            rows=per_class_rows,
            output_path=per_class_csv,
        )

    if (
        REQUIRE_ALL_FINAL_RUNS
        and len(final_rows) != len(FINAL_EVALUATION_SEEDS)
    ):
        raise RuntimeError(
            f"Expected {len(FINAL_EVALUATION_SEEDS)} runs, "
            f"got {len(final_rows)}."
        )

    summary_metric_names = [
        "val_precision",
        "val_recall",
        "val_map50",
        "val_map50_95",
        "val_bottom3_mean_map50_95",
        "test_precision",
        "test_recall",
        "test_map50",
        "test_map50_95",
        "test_bottom3_mean_map50_95",
        "training_seconds",
        "validation_seconds",
        "test_seconds",
        "run_wall_seconds",
    ]

    summary_rows = summarize_numeric_rows(
        rows=final_rows,
        metric_names=summary_metric_names,
    )

    summary_rows.append({
        "metric": "baseline_epoch_units",
        "n": 1,
        "mean": int(BASELINE_EPOCH_UNITS),
        "sd": 0.0,
        "min": int(BASELINE_EPOCH_UNITS),
        "max": int(BASELINE_EPOCH_UNITS),
    })

    save_rows_to_csv(
        rows=summary_rows,
        output_path=final_summary_csv,
    )

    per_class_summary_rows: List[Dict[str, Any]] = []

    for class_name in CLASS_NAMES:
        values = [
            float(row["test_map50_95"])
            for row in per_class_rows
            if row["class_name"] == class_name
        ]

        if not values:
            raise RuntimeError(
                f"No test per class metric was saved for {class_name}."
            )

        per_class_summary_rows.append({
            "class_name": class_name,
            "n": len(values),
            "mean_test_map50_95": float(mean(values)),
            "sd_test_map50_95": (
                float(stdev(values))
                if len(values) > 1
                else 0.0
            ),
            "min_test_map50_95": float(min(values)),
            "max_test_map50_95": float(max(values)),
        })

    save_rows_to_csv(
        rows=per_class_summary_rows,
        output_path=per_class_summary_csv,
    )

    plot_training_trajectories(
        final_rows=final_rows,
        output_path=(
            RUN_PLOTS_ROOT
            / "untuned_yolov8m_training_trajectories.png"
        ),
    )

    plot_test_metric_summary(
        final_rows=final_rows,
        output_path=(
            RUN_PLOTS_ROOT
            / "untuned_yolov8m_test_metric_summary.png"
        ),
    )

    workflow_seconds = time.perf_counter() - workflow_start

    manifest = {
        "method": "Untuned YOLOv8m baseline",
        "dataset_tag": DATASET_TAG,
        "source_dataset_root": str(
            SOURCE_DATASET_ROOT.resolve()
        ),
        "model_name": MODEL_NAME,
        "split_ratios": {
            "train": TRAIN_RATIO,
            "val": VALID_RATIO,
            "test": TEST_RATIO,
        },
        "seeds": list(FINAL_EVALUATION_SEEDS),
        "epochs_per_run": int(FULL_FIDELITY_EPOCHS),
        "baseline_epoch_units": int(BASELINE_EPOCH_UNITS),
        "imgsz": int(IMAGE_SIZE),
        "batch": int(BATCH_SIZE),
        "workers": int(WORKERS),
        "patience": int(PATIENCE),
        "cache": CACHE_MODE,
        "image_placement_mode": IMAGE_PLACEMENT_MODE,
        "untuned_hyperparameters": dict(
            UNTUNED_HYPERPARAMETERS
        ),
        "fixed_training_args": dict(FIXED_TRAINING_ARGS),
        "test_used_for_selection": False,
        "device": device,
        "workflow_seconds": float(workflow_seconds),
        "result_files": {
            "three_seed_results": str(
                final_results_csv.resolve()
            ),
            "mean_sd_summary": str(
                final_summary_csv.resolve()
            ),
            "per_class_results": str(
                per_class_csv.resolve()
            ),
            "per_class_summary": str(
                per_class_summary_csv.resolve()
            ),
        },
    }

    manifest_json.write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    print("\nFINAL THREE SEED TEST SUMMARY")
    for row in summary_rows:
        if str(row["metric"]).startswith("test_"):
            print(
                f"{row['metric']:30s} "
                f"mean={float(row['mean']):.6f} "
                f"sd={float(row['sd']):.6f}"
            )

    print("\nSaved results:", final_results_csv.resolve())
    print("Saved summary:", final_summary_csv.resolve())
    print("Saved manifest:", manifest_json.resolve())

    return {
        "status": "PASS",
        "final_rows": final_rows,
        "summary_rows": summary_rows,
        "per_class_rows": per_class_rows,
        "per_class_summary_rows": per_class_summary_rows,
        "manifest": manifest,
    }


## Module 10: Run switch and cleanup

Set `RUN_UNTUNED_BASELINE` to `True` for the complete experiment. Cleanup runs only after all three simulations finish successfully. The temporary `yolo_data` folder is removed, and any HPO folder inside this baseline run is also removed if it exists. Training outputs, checkpoints, statistics, plots, and split manifests are preserved.


In [ ]:
# Keep False while reviewing the notebook.
# Set True for the complete three seed baseline.
RUN_UNTUNED_BASELINE = False

if RUN_UNTUNED_BASELINE:
    untuned_baseline_result = run_untuned_yolov8m_baseline(
        all_records=records,
    )

    import shutil

    if YOLO_DATA_ROOT.exists():
        shutil.rmtree(YOLO_DATA_ROOT)
        print(f"Deleted temporary YOLO data: {YOLO_DATA_ROOT}")

    if HPO_RUNS_ROOT.exists():
        shutil.rmtree(HPO_RUNS_ROOT)
        print(f"Deleted HPO folder: {HPO_RUNS_ROOT}")

else:
    print("RUN_UNTUNED_BASELINE is False.")
    print("Set it to True when ready to run all three seeds.")
    print("Model:", MODEL_NAME)
    print("Dataset:", DATASET_TAG)
    print("Seeds:", FINAL_EVALUATION_SEEDS)
    print("Epochs per run:", FULL_FIDELITY_EPOCHS)
    print("Total baseline epoch units:", BASELINE_EPOCH_UNITS)
    print("Workers / cache:", WORKERS, "/", CACHE_MODE)
    print("Patience:", PATIENCE)
    print("Test split is used only after each model has been trained.")


## Final protocol safeguards

1. Model scale is fixed to YOLOv8m.
2. The baseline hyperparameters are the untuned reference values used by the newer optimization notebook.
3. Each seed uses a new deterministic 70/15/15 split.
4. Train, validation, and test filenames are checked for overlap.
5. The test split cannot influence training or checkpoint selection.
6. Every final run uses 40 epochs, matching the full fidelity training budget used by the optimization methods.
7. Three final seeds produce 120 baseline training epoch units.
8. Temporary YOLO formatted data is deleted only after the complete workflow succeeds.
